# Gradient Norm Sanity Check

Plot per-layer gradient L2 norms during training. Useful for catching vanishing or exploding gradients early.

In [ ]:
import torch
import torch.nn as nn
from collections import defaultdict
import matplotlib.pyplot as plt

In [ ]:
def grad_norms(model):
    out = {}
    for name, p in model.named_parameters():
        if p.grad is not None:
            out[name] = p.grad.detach().norm().item()
    return out

In [ ]:
torch.manual_seed(0)
# Deeper network -- gradient pathologies show up more clearly
model = nn.Sequential(
    *[layer for _ in range(8) for layer in (nn.Linear(64, 64), nn.Tanh())],
    nn.Linear(64, 2),
)
X = torch.randn(64, 64); y = torch.randint(0, 2, (64,))
criterion = nn.CrossEntropyLoss()
history = defaultdict(list)
for step in range(50):
    out = model(X); loss = criterion(out, y)
    model.zero_grad(); loss.backward()
    for k, v in grad_norms(model).items():
        history[k].append(v)
    for p in model.parameters():
        if p.grad is not None:
            p.data.add_(p.grad, alpha=-1e-2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, values in history.items():
    if 'weight' in name:
        ax.plot(values, label=name, alpha=0.6)
ax.set_yscale('log')
ax.set_xlabel('step')
ax.set_ylabel('grad L2 norm')
ax.set_title('Per-layer gradient norms during training')
ax.grid(alpha=0.3)
ax.legend(fontsize=7, ncol=2)
plt.show()

## What to watch for

- **Vanishing**: early-layer gradients drift toward 0 over training.
- **Exploding**: any layer's norm spikes by 10x or more from one step to the next.
- **Healthy**: all layers stay within ~1 order of magnitude of each other.